## Exploring and Preprocessing Mental Health Data

### Schritt 1: Initialisierung und Laden der Daten
- Importiere die erforderlichen Bibliotheken und lade die Trainings- und Testdatensätze.
- Verarbeite beide Datensätze gleichzeitig, damit das endgültige Modell bei der Vorhersage genau dieselben Merkmalsstrukturen erhält.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Load datasets
train_dataset = pd.read_csv("train.csv")
test_dataset = pd.read_csv("test.csv")

display(train_dataset.head())

,id,Name,Gender,Age,City,Working Professional or Student,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Depression
0,0,Aaradhya,Female,49.0,Ludhiana,Working Professional,Chef,NaN,5.0,NaN,NaN,2.0,More than 8 hours,Healthy,BHM,No,1.0,2.0,No,0
1,1,Vivan,Male,26.0,Varanasi,Working Professional,Teacher,NaN,4.0,NaN,NaN,3.0,Less than 5 hours,Unhealthy,LLB,Yes,7.0,3.0,No,1
2,2,Yuvraj,Male,33.0,Visakhapatnam,Student,NaN,5.0,NaN,8.97,2.0,NaN,5-6 hours,Healthy,B.Pharm,Yes,3.0,1.0,No,1
3,3,Yuvraj,Male,22.0,Mumbai,Working Professional,Teacher,NaN,5.0,NaN,NaN,1.0,Less than 5 hours,Moderate,BBA,Yes,10.0,1.0,Yes,1
4,4,Rhea,Female,30.0,Kanpur,Working Professional,Business Analyst,NaN,1.0,NaN,NaN,1.0,5-6 hours,Unhealthy,BBA,Yes,9.0,4.0,Yes,0


### Schritt 2: Feature Engineering & Datenbereinigung
Es gibt mehrere Spalten, die bereinigt und zusammengefasst werden müssen. Um zu vermeiden, denselben Code zweimal zu schreiben (einmal für das Training, einmal für den Test), erstellen wir eine einzige Funktion `preprocess_features`, die die gesamte Logik abdeckt:

1. **Kombinierter Druck:** `Academic Pressure` und `Work Pressure` schließen sich gegenseitig aus => zusammenführen
2. **Kombinierte Zufriedenheit:** Ebenso werden wir `Study Satisfaction` und `Job Satisfaction` zusammenführen.
3. **Studentenstatus:** Umwandlung von `Working Professional or Student` in eine binäre Spalte `is_student`.
4. **Gesundheitsindex:** Zuordnung von `Dietary Habits` und `Sleep Duration` zu numerischen Werten und Zusammenführung zu einem einzigen Index.
5. **Umgang mit fehlenden Werten:** Auffüllen von NaNs in numerischen Spalten (wie CGPA und Alter) und kategorialen Spalten (wie Beruf und Abschluss).
6. **Entfernen unnötiger Daten:** Wir entfernen `Name`, da dieser keinen prädiktiven Wert hat.


In [ ]:
def preprocess_features(df):
    df = df.copy()
    
    # 1. Kombinierter Druck
    df['Academic Pressure'] = df['Academic Pressure'].fillna(0)
    df['Work Pressure'] = df['Work Pressure'].fillna(0)
    df['Combined Pressure'] = df['Academic Pressure'] + df['Work Pressure']
    df.drop(['Academic Pressure', 'Work Pressure'], axis=1, inplace=True)
    
    # 2. Kombinierter Zufriedenheit
    df['Study Satisfaction'] = df['Study Satisfaction'].fillna(0)
    df['Job Satisfaction'] = df['Job Satisfaction'].fillna(0)
    df['Combined Satisfaction'] = df['Study Satisfaction'] + df['Job Satisfaction']
    df.drop(['Study Satisfaction', 'Job Satisfaction'], axis=1, inplace=True)
    
    # 3. Studentenstatus
    df['is_student'] = (df['Working Professional or Student'] == 'Student').astype(int)
    df.drop('Working Professional or Student', axis=1, inplace=True)
    
    # 4. Gesundheitsindex
    diet_map = {'Healthy': 3, 'Moderate': 2, 'Unhealthy': 1}
    sleep_map = {'More than 8 hours': 4, '7-8 hours': 3, '5-6 hours': 2, 'Less than 5 hours': 1}
    df['Dietary Habits'] = df['Dietary Habits'].fillna('Moderate')
    df['Sleep Duration'] = df['Sleep Duration'].fillna('7-8 hours')
    df['Health_Index'] = df['Dietary Habits'].map(diet_map) + df['Sleep Duration'].map(sleep_map)
    df.drop(['Dietary Habits', 'Sleep Duration'], axis=1, inplace=True)
    
    # 5. Umgang mit fehlenden Werten
    df['CGPA'] = df['CGPA'].fillna(0)
    df['Age'] = df['Age'].fillna(df['Age'].median())
    df['Work/Study Hours'] = df['Work/Study Hours'].fillna(df['Work/Study Hours'].median())
    df['Financial Stress'] = df['Financial Stress'].fillna(0)
    df['Profession'] = df['Profession'].fillna('Unknown')
    df['Degree'] = df['Degree'].fillna('Unknown')
    df['City'] = df['City'].fillna('Unknown')
    df['Gender'] = df['Gender'].fillna('Unknown')
    df['Have you ever had suicidal thoughts ?'] = df['Have you ever had suicidal thoughts ?'].fillna('Unknown')
    df['Family History of Mental Illness'] = df['Family History of Mental Illness'].fillna('Unknown')
    
    # 6. Entfernen unnötiger daten (Name)
    df.drop(['Name'], axis=1, inplace=True)
    
    return df

X_train_full = preprocess_features(train_dataset)
X_test = preprocess_features(test_dataset)

y_train = X_train_full['Depression']    # target vector
X_train = X_train_full.drop(['Depression'], axis=1)    # feature matrix

display(X_train.head())

### Schritt 3: Kodierung kategorialer Variablen
- Umwandlung der verbleibenden Textspalten (`Gender`, `City`, `Profession`, `Degree`, `Have you ever had suicidal thoughts ?`, `Family History of Mental Illness`) in Zahlen. 
- Dazu werden `ColumnTransformer` und `OneHotEncoder` aus Scikit-Learn verwendet, um dies dynamisch durchzuführen.
- Skalierung der numerischen Merkmale, damit sie auf derselben Skala liegen.

In [3]:
categorical_cols = ['Gender', 'City', 'Profession', 'Degree', 
                    'Have you ever had suicidal thoughts ?', 'Family History of Mental Illness']

numerical_cols = ['Age', 'CGPA', 'Work/Study Hours', 'Financial Stress', 
                  'Combined Pressure', 'Combined Satisfaction', 'is_student', 'Health_Index']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ],
    remainder='passthrough'
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(X_train)

            id  Gender   Age           City         Profession  CGPA  \
0            0  Female  49.0       Ludhiana               Chef  0.00   
1            1    Male  26.0       Varanasi            Teacher  0.00   
2            2    Male  33.0  Visakhapatnam            Unknown  8.97   
3            3    Male  22.0         Mumbai            Teacher  0.00   
4            4  Female  30.0         Kanpur   Business Analyst  0.00   
...        ...     ...   ...            ...                ...   ...   
140695  140695  Female  18.0      Ahmedabad            Unknown  0.00   
140696  140696  Female  41.0      Hyderabad     Content Writer  0.00   
140697  140697  Female  24.0        Kolkata  Marketing Manager  0.00   
140698  140698  Female  49.0       Srinagar            Plumber  0.00   
140699  140699    Male  27.0          Patna            Unknown  9.24   

          Degree Have you ever had suicidal thoughts ?  Work/Study Hours  \
0            BHM                                    No     

## Splitting Training Data 
75% **training** - 25% **test** Data with **cross-validation**
every teammember has to have the same split, so that the results from the models are comparable  

In [5]:
# Everyone uses this exact split
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_processed, y_train,
    test_size=0.25,
    random_state=42,    # this number must be the same for all 3 of us
    stratify=y_train    # ensures same class ratio in both splits
)

NameError: name 'train_test_split' is not defined

## Plug-in example for each model
this is an example of how your model will be plugged in(**don't run this code**):

In [ ]:
model = SomeClassifier()   # swap this out per person
model.fit(X_tr, y_tr)
print(model.score(X_val, y_val))

## Actual implementation of the plug in:

In [ ]:
# Model 1 Diana
rf = RandomForestClassifier()
rf.fit(X_tr, y_tr)
print("Random Forest:", rf.score(X_val, y_val))

# Model 2 Diana
xgb = XGBClassifier()
xgb.fit(X_tr, y_tr)
print("XGBoost:", xgb.score(X_val, y_val))


## Hyperparameter Tuning with RandomizedSearchCV

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report

# --- Random Forest ---
rf_param_grid = {
    'n_estimators':      [100, 200, 300],
    'max_depth':         [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'max_features':      ['sqrt', 'log2', 0.3],
    'class_weight':      ['balanced', {0: 1, 1: 3}, {0: 1, 1: 5}]
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=rf_param_grid,
    n_iter=20,
    scoring='f1',       # optimises for class 1 (depression) F1
    cv=3,
    random_state=42,
    n_jobs=-1           # use all CPU cores to speed it up
)

rf_search.fit(X_tr, y_tr)

print("Best RF params:", rf_search.best_params_)
print("Best RF F1 score (CV):", rf_search.best_score_)

y_pred_rf_tuned = rf_search.best_estimator_.predict(X_val)
print(classification_report(y_val, y_pred_rf_tuned))

In [ ]:
# --- XGBoost ---
xgb_param_grid = {
    'n_estimators':      [100, 200, 300],
    'max_depth':         [3, 5, 7],
    'learning_rate':     [0.05, 0.1, 0.2],
    'subsample':         [0.7, 0.8, 1.0],
    'scale_pos_weight':  [1, 3, 4.5]   # 4.5 ≈ class 0 count / class 1 count
}

xgb_search = RandomizedSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss'),
    param_distributions=xgb_param_grid,
    n_iter=20,
    scoring='f1',
    cv=3,
    random_state=42,
    n_jobs=-1
)

xgb_search.fit(X_tr, y_tr)

print("Best XGB params:", xgb_search.best_params_)
print("Best XGB F1 score (CV):", xgb_search.best_score_)

y_pred_xgb_tuned = xgb_search.best_estimator_.predict(X_val)
print(classification_report(y_val, y_pred_xgb_tuned))